In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')


# TESLA DELIVERIES FORECASTING PROJECT

# 1. IMPORT LIBRARIES


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    TimeSeriesSplit,
    GridSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.linear_model import Ridge

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from statsmodels.tsa.stattools import adfuller

# 2. LOAD DATASET
The dataset was loaded using Pandas and inspected for:

**Missing values**:

**Duplicate records**:

**Data types**:

**Statistical summaries**:

In [ ]:
# ==========================
# 2. LOAD DATASET
# ==========================

dataset_path = "/kaggle/input/datasets/nalisha/tesla-ea-deliveries-and-production-data20152025/tesla_deliveries_dataset_2015_2025.csv"

df = pd.read_csv(dataset_path)

print("Dataset Shape:", df.shape)
display(df.head())

# 3. BASIC DATA CHECK

In this step, the dataset is inspected to understand its overall structure and quality before applying preprocessing and machine learning techniques.

The following checks are performed:

- Dataset information using `df.info()`
- Missing value analysis using `df.isnull().sum()`
- Statistical summary using `df.describe()`

These checks help identify:
- Data types of features
- Presence of null values
- Distribution of numerical features
- Mean, median, minimum, maximum, and standard deviation values

Performing these checks is important for ensuring data consistency and preparing the dataset for further analysis and modeling.


In [ ]:
# ==========================
# 3. BASIC DATA CHECK
# ==========================

print("\nDataset Info:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nStatistical Summary:")
display(df.describe())


# Handling Missing Values

In this step, missing values in numerical columns are handled using median imputation.

The following numerical features are considered:

- Estimated_Deliveries
- Production_Units
- Avg_Price_USD
- Battery_Capacity_kWh
- Range_km
- CO2_Saved_tons
- Charging_Stations

Median imputation is used because it is less sensitive to outliers and helps maintain the stability of the dataset.

This preprocessing step ensures that the Machine Learning model receives clean and complete data for training and evaluation.


In [ ]:
# ==========================
# 4. HANDLE MISSING VALUES
# ==========================

numerical_cols = [
    'Estimated_Deliveries',
    'Production_Units',
    'Avg_Price_USD',
    'Battery_Capacity_kWh',
    'Range_km',
    'CO2_Saved_tons',
    'Charging_Stations'
]

for col in numerical_cols:
    df[col].fillna(df[col].median(), inplace=True)

# 5. FEATURE ENGINEERING

Feature Engineering is performed to create additional meaningful features that help improve the forecasting performance of the Machine Learning model.

## Date Feature Creation
A new `Date` column is created by combining the `Year` and `Month` columns. This enables proper time-series analysis and chronological sorting of the dataset.

## Time-Based Features
The following temporal features are extracted from the `Date` column:

- Quarter
- Week_Of_Year

These features help the model capture seasonal and yearly delivery trends.

## Lag Features
Lag features are created to provide historical delivery information to the model.

- Lag_1 → Previous delivery value
- Lag_2 → Delivery value from two previous observations

Lag features are important in time-series forecasting because past values strongly influence future predictions.

## Rolling Statistics
Rolling statistics are used to capture short-term delivery trends and variability.

- Rolling_Mean_3 → Average deliveries over the last 3 observations
- Rolling_STD_3 → Standard deviation over the last 3 observations

These features help the model understand moving trends and fluctuations in Tesla deliveries.

## Handling Missing Values After Feature Engineering
Lag and rolling operations create missing values at the beginning of the dataset. Backward filling (`bfill`) is applied to handle these missing values and maintain dataset consistency.



In [ ]:
# ==========================
# 5. FEATURE ENGINEERING
# ==========================

df["Date"] = pd.to_datetime(
    df["Year"].astype(str)
    + "-"
    + df["Month"].astype(str)
)

df = df.sort_values("Date")

# Time Features
df["Quarter"] = df["Date"].dt.quarter
df["Week_Of_Year"] = df["Date"].dt.isocalendar().week.astype(int)

# Lag Features
df["Lag_1"] = df["Estimated_Deliveries"].shift(1)
df["Lag_2"] = df["Estimated_Deliveries"].shift(2)

# Rolling Statistics
df["Rolling_Mean_3"] = (
    df["Estimated_Deliveries"]
    .rolling(3)
    .mean()
)

df["Rolling_STD_3"] = (
    df["Estimated_Deliveries"]
    .rolling(3)
    .std()
)

df = df.bfill()

# 6.Exploratory Data Analysis (EDA)

Exploratory Data Analysis is performed to better understand the relationships, patterns, and trends present in the Tesla deliveries dataset.

## Correlation Heatmap
A correlation heatmap is generated to analyze the relationships between numerical features in the dataset.

The heatmap helps identify:
- Strong positive correlations
- Negative correlations
- Feature dependencies
- Important variables affecting Tesla deliveries

This analysis is useful for understanding how different factors such as production units, battery capacity, vehicle range, and CO₂ savings influence estimated deliveries.

## Tesla Deliveries Trend Analysis
A yearly delivery trend visualization is created to observe how Tesla deliveries changed over time from 2015 to 2025.

The line plot helps identify:
- Growth trends
- Seasonal patterns
- Delivery fluctuations
- Long-term business expansion

This analysis provides valuable insights into Tesla’s market growth and production performance over the years.
```


In [ ]:
# ==========================
# 6. EDA
# ==========================

plt.figure(figsize=(12,8))

sns.heatmap(
    df.corr(numeric_only=True),
    cmap="coolwarm",
    annot=True
)

plt.title("Correlation Heatmap")
plt.show()

# Delivery Trend

trend = (
    df.groupby("Year")["Estimated_Deliveries"]
    .sum()
)
plt.figure(figsize=(10,5))

trend.plot(marker="o")

plt.title("Tesla Deliveries Trend")
plt.ylabel("Estimated Deliveries")
plt.grid(True)

plt.show()

# 7. TRAIN TEST SPLIT

The dataset is divided into training and testing sets to evaluate the Machine Learning model's performance on unseen data.

## Target Variable
The target variable selected for prediction is:

- `Estimated_Deliveries`

This variable represents the estimated number of Tesla vehicle deliveries.

## Feature Selection
All relevant features are selected as input variables (`X`) except:
- Estimated_Deliveries (target)
- Date column

The `Date` column is excluded because time-related information has already been extracted through feature engineering.

## Chronological Splitting
Instead of using a random split, a chronological split is applied to preserve the time-series nature of the dataset.

- 80% of the data is used for training
- 20% of the data is used for testing

This approach prevents future data leakage and ensures that the model is trained only on past observations while being tested on future observations.

Chronological splitting is considered a best practice for time-series forecasting problems.



In [ ]:
# ==========================
# 7. TRAIN TEST SPLIT
# ==========================

target = "Estimated_Deliveries"

X = df.drop(
    columns=[
        target,
        "Date"
    ]
)

y = df[target]

split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]


# 8. PREPROCESSING PIPELINE

A preprocessing pipeline is created to automate data preparation before model training.

## Numerical Features
Numerical columns are processed using:
- Median Imputation
- StandardScaler

## Categorical Features
Categorical columns are processed using:
- Most Frequent Imputation
- OneHotEncoder

## ColumnTransformer
`ColumnTransformer` combines numerical and categorical preprocessing into a single workflow.

This helps:
- Prevent data leakage
- Maintain consistency
- Improve model performance

In [ ]:
# ==========================
# 8. PREPROCESSING PIPELINE
# ==========================

categorical_cols = X.select_dtypes(
    include=["object"]
).columns

numeric_cols = X.select_dtypes(
    exclude=["object"]
).columns

numeric_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

categorical_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])

preprocessor = ColumnTransformer([
    (
        "num",
        numeric_transformer,
        numeric_cols
    ),
    (
        "cat",
        categorical_transformer,
        categorical_cols
    )
])


# 9. RIDGE MODEL

A Ridge Regression model is implemented using a Machine Learning pipeline.

The pipeline combines:
- Data preprocessing
- Feature transformation
- Model training

Ridge Regression uses L2 regularization to reduce overfitting and improve model generalization.

This model is well-suited for structured tabular datasets and performs effectively for regression-based forecasting tasks.


In [ ]:
# ==========================
# 9. RIDGE MODEL
# ==========================

ridge_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        Ridge()
    )
])


# 10. HYPERPARAMETER TUNING

Hyperparameter tuning is performed using `GridSearchCV` to find the best value of the Ridge Regression alpha parameter.

## TimeSeriesSplit
`TimeSeriesSplit` is used for cross-validation to preserve the chronological order of time-series data and prevent future data leakage.

## GridSearchCV
Different alpha values are tested to identify the best model configuration based on the R² score.

This process helps improve:
- Model accuracy
- Generalization performance
- Forecasting capability

In [ ]:
# ==========================
# 10. HYPERPARAMETER TUNING
# ==========================

param_grid = {
    "model__alpha": [
        0.01,
        0.1,
        1,
        10,
        50,
        100
    ]
}

tscv = TimeSeriesSplit(
    n_splits=5
)
grid = GridSearchCV(
    ridge_pipeline,
    param_grid,
    cv=tscv,
    scoring="r2"
)

grid.fit(
    X_train,
    y_train
)

print("Best Parameters:")
print(grid.best_params_)

# 11. MODEL PREDICTION

The best Ridge Regression model obtained from GridSearchCV is selected for prediction.

The trained model is used to generate predictions on the test dataset.

These predictions are later evaluated using performance metrics such as:
- MAE
- RMSE
- R² Score

This step helps measure how accurately the model forecasts Tesla vehicle deliveries.

In [ ]:
# ==========================
# 11. MODEL PREDICTION
# ==========================

best_model = grid.best_estimator_

preds = best_model.predict(
    X_test
)

# 12. EVALUATION


The performance of the Ridge Regression model is evaluated using standard regression metrics.

## Evaluation Metrics

- MAE (Mean Absolute Error)
- RMSE (Root Mean Squared Error)
- R² Score

These metrics help measure:
- Prediction accuracy
- Error magnitude
- Overall model performance

The obtained results indicate that the model performs exceptionally well in forecasting Tesla vehicle deliveries.

In [ ]:
# ==========================
# 12. EVALUATION
# ==========================

mae = mean_absolute_error(
    y_test,
    preds
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        preds
    )
)

r2 = r2_score(
    y_test,
    preds
)
print("\nRESULTS")
print("="*40)

print("MAE :", round(mae,2))
print("RMSE :", round(rmse,2))
print("R2 :", round(r2,5))

# 13. ACTUAL VS PREDICTED

A comparison plot is created to visualize the difference between actual Tesla deliveries and predicted deliveries generated by the Ridge Regression model.

The graph helps evaluate:
- Prediction accuracy
- Forecasting performance
- Similarity between actual and predicted trends

A close alignment between the two curves indicates that the model is performing effectively and generating highly accurate forecasts.

In [ ]:
# ==========================
# 13. ACTUAL VS PREDICTED
# ==========================

plt.figure(figsize=(12,6))

plt.plot(
    y_test.values[:100],
    label="Actual"
)

plt.plot(
    preds[:100],
    label="Predicted"
)

plt.title(
    "Actual vs Predicted Deliveries"
)

plt.xlabel("Observations")
plt.ylabel("Deliveries")
plt.legend()
plt.grid(True)

plt.show()


# 14. RESIDUAL ANALYSIS

Residual analysis is performed to examine the prediction errors of the Ridge Regression model.

Residuals represent the difference between:
- Actual values
- Predicted values

The residual distribution plot helps analyze:
- Error patterns
- Model stability
- Prediction consistency

A well-centered residual distribution around zero indicates that the model predictions are accurate and unbiased.

In [ ]:
# ==========================
# 14. RESIDUAL ANALYSIS
# ==========================

residuals = y_test - preds

plt.figure(figsize=(10,6))

sns.histplot(
    residuals,
    bins=30,
    kde=True
)

plt.title("Residual Distribution")

plt.show()

# 15. STATIONARITY TEST

The Augmented Dickey-Fuller (ADF) Test is performed to check whether the Tesla deliveries time series is stationary.

Stationarity is important in time-series forecasting because a stationary series has:
- Constant mean
- Constant variance
- Stable patterns over time

## ADF Test Results
- ADF Statistic
- P-Value

If the P-Value is less than 0.05, the series is considered stationary.

The obtained results indicate that the Tesla deliveries series is stationary and suitable for forecasting analysis.


In [ ]:
# ==========================
# 15. STATIONARITY TEST
# ==========================

result = adfuller(
    df["Estimated_Deliveries"]
)

print("\nADF Statistic:", result[0])
print("P-Value:", result[1])

if result[1] < 0.05:
    print("Series is Stationary")
else:
    print("Series is Non-Stationary")


# 16. FORECAST TABLE

A forecast table is created to compare the actual Tesla deliveries with the predicted deliveries generated by the Ridge Regression model.

This comparison helps:
- Evaluate prediction accuracy
- Analyze forecasting performance
- Understand the closeness between actual and predicted values

The displayed sample forecasts demonstrate that the model is able to generate highly accurate delivery predictions.

In [ ]:
# ==========================
# 16. FORECAST TABLE
# ==========================

forecast_df = pd.DataFrame({
    "Actual": y_test.values[:20],
    "Predicted": preds[:20]
})

print("\nSample Forecasts:")
display(forecast_df)

# 17. FINAL CONCLUSION

The Tesla Deliveries Forecasting project successfully demonstrates the implementation of a complete end-to-end Machine Learning pipeline for time-series forecasting.

The project included:
- Data preprocessing
- Exploratory Data Analysis (EDA)
- Feature engineering
- Preprocessing pipeline creation
- Hyperparameter tuning
- Ridge Regression modeling
- Forecast evaluation

The Ridge Regression model achieved excellent performance with:
- Very low prediction error
- High forecasting accuracy
- Strong R² score

The results indicate that the model is highly effective in predicting Tesla vehicle deliveries using historical production, pricing, battery, and sustainability-related features.

This project highlights the practical application of Machine Learning and time-series analysis in the electric vehicle industry for forecasting and business trend analysis.

In [ ]:
# ==========================
# 17. FINAL CONCLUSION
# ==========================

print("\nPROJECT SUMMARY")
print("="*40)

print("Best Alpha:", grid.best_params_["model__alpha"])
print("MAE :", round(mae,2))
print("RMSE :", round(rmse,2))
print("R2 Score :", round(r2,5))